In [1]:
import os
import glob
import pickle
from fingerprint import build_database

song_files = glob.glob('songs/*.mp3')
print(f"Found {len(song_files)} songs")

database = build_database(song_files, sr=22050)

with open('database.pkl', 'wb') as f:
    pickle.dump(database, f)

print(f"Database saved! {len(database)} unique hashes")

Found 50 songs
Indexing: A Day In The Life...
  → 4144 hashes generated
Indexing: A Hard Day_s Night...
  → 6037 hashes generated
Indexing: Across The Universe...
  → 4685 hashes generated
Indexing: Back In The U.S.S.R....
  → 3302 hashes generated
Indexing: Blackbird...
  → 1668 hashes generated
Indexing: Bohemian Rhapsody...
  → 3406 hashes generated
Indexing: Can_t Buy Me Love...
  → 4936 hashes generated
Indexing: Crazy Little Thing Called Love...
  → 3826 hashes generated
Indexing: Day Tripper...
  → 4857 hashes generated
Indexing: Don_t Stop Me Now...
  → 6384 hashes generated
Indexing: Drive My Car...
  → 3486 hashes generated
Indexing: Eight Days A Week...
  → 5592 hashes generated
Indexing: Eleanor Rigby...
  → 3984 hashes generated
Indexing: Get Back...
  → 4414 hashes generated
Indexing: Hello, Goodbye...
  → 7772 hashes generated
Indexing: Help!...
  → 4324 hashes generated
Indexing: Helter Skelter...
  → 6748 hashes generated
Indexing: Hey Jude...
  → 9047 hashes generated

In [2]:
import librosa

In [7]:
import glob

In [8]:
song_files = glob.glob('songs/*.mp3')

In [9]:
from mutagen.mp3 import MP3
from mutagen.id3 import ID3, TIT2, TPE1

def get_song_metadata(song_path):
    """Extract title and artist from mp3 tags"""
    try:
        tags = ID3(song_path)
        title  = tags.get('TIT2', None)
        artist = tags.get('TPE1', None)
        title  = str(title)  if title  else os.path.basename(song_path).replace('.mp3', '')
        artist = str(artist) if artist else 'Unknown Artist'
    except Exception:
        title  = os.path.basename(song_path).replace('.mp3', '')
        artist = 'Unknown Artist'
    return title, artist

# verify it works
for song_path in song_files[:3]:
    title, artist = get_song_metadata(song_path)
    print(f"{song_path} → '{title}' by '{artist}'")

songs\A Day In The Life.mp3 → 'A Day In The Life - Remastered' by 'The Beatles'
songs\A Hard Day_s Night.mp3 → 'A Hard Day's Night - Remastered' by 'The Beatles'
songs\Across The Universe.mp3 → 'Across The Universe - Remastered' by 'The Beatles'


In [10]:
def build_database_with_metadata(song_files, sr=22050):
    database  = {}
    metadata  = {}  # song_name -> (title, artist)

    for song_path in song_files:
        song_name = os.path.basename(song_path).replace('.mp3', '')
        title, artist = get_song_metadata(song_path)
        metadata[song_name] = (title, artist)
        
        print(f"Indexing: {title} by {artist}...")
        y, _ = librosa.load(song_path, sr=sr, mono=True)
        peaks_list, f, t = get_peaks(y, sr)
        hashes = generate_hashes(peaks_list, f, t)
        
        for hash_key, t_idx in hashes:
            if hash_key not in database:
                database[hash_key] = []
            database[hash_key].append((song_name, t_idx))
        
        print(f"  → {len(hashes)} hashes generated")
    
    return database, metadata

In [12]:
import os

In [13]:
database, metadata = build_database_with_metadata(song_files, sr=22050)

with open('database.pkl', 'wb') as f:
    pickle.dump({'database': database, 'metadata': metadata}, f)

print(f"Saved: {len(database)} hashes, {len(metadata)} songs")

# verify metadata
for song_name, (title, artist) in list(metadata.items())[:5]:
    print(f"  {song_name} → '{title}' by '{artist}'")

Indexing: A Day In The Life - Remastered by The Beatles...


NameError: name 'get_peaks' is not defined

In [2]:
# Load Blackbird
y, sr = librosa.load('Blackbird.mp3', sr=22050, mono=True)

# Get peaks and hashes twice
p1, f1, t1 = get_peaks(y, sr)
p2, f2, t2 = get_peaks(y, sr)

h1 = generate_hashes(p1, f1, t1)
h2 = generate_hashes(p2, f2, t2)

print("Peaks same:", p1 == p2)
print("Hashes same:", h1[:5] == h2[:5])
print("Sample hash type:", type(h1[0][0]))
print("Sample hashes:", h1[:3])

# Check what's in database for Blackbird
all_songs = list(set(name for entries in database.values() for name, _ in entries))
print("\nSongs in database:", all_songs)

# Count how many Blackbird hashes exist
blackbird_hashes = sum(
    1 for entries in database.values()
    for name, _ in entries if name == 'Blackbird'
)
print(f"Blackbird hashes in database: {blackbird_hashes}")

# Check overlap with query
matches = sum(1 for hk, _ in h1 if hk in database)
print(f"Query hashes found in database: {matches}")

NameError: name 'librosa' is not defined

In [4]:
!pip install mutagen



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# build_artist_db.py
import json
import os
import glob
from mutagen.id3 import ID3

def build_artist_database(song_files):
    """Extract metadata from mp3 tags and save as JSON"""
    artist_db = {}
    
    for song_path in song_files:
        song_name = os.path.basename(song_path).replace('.mp3', '')
        try:
            tags = ID3(song_path)
            title  = str(tags['TIT2']) if 'TIT2' in tags else song_name
            artist = str(tags['TPE1']) if 'TPE1' in tags else 'Unknown Artist'
            album  = str(tags['TALB']) if 'TALB' in tags else 'Unknown Album'
            year   = str(tags['TDRC']) if 'TDRC' in tags else 'Unknown Year'
        except Exception:
            title  = song_name
            artist = 'Unknown Artist'
            album  = 'Unknown Album'
            year   = 'Unknown Year'
        
        artist_db[song_name] = {
            'title':  title,
            'artist': artist,
            'album':  album,
            'year':   year
        }
        print(f"  {song_name} → '{title}' by '{artist}' ({album}, {year})")
    
    with open('artist_db.json', 'w') as f:
        json.dump(artist_db, f, indent=2)
    
    print(f"\nSaved artist_db.json with {len(artist_db)} entries")
    return artist_db

# run it
song_files = glob.glob('songs/*.mp3')
print(f"Found {len(song_files)} songs\n")
artist_db = build_artist_database(song_files)

Found 50 songs

  A Day In The Life → 'A Day In The Life - Remastered' by 'The Beatles' (Sgt. Pepper's Lonely Hearts Club Band (Remastered), 1967)
  A Hard Day_s Night → 'A Hard Day's Night - Remastered' by 'The Beatles' (A Hard Day's Night (Remastered), 1964)
  Across The Universe → 'Across The Universe - Remastered' by 'The Beatles' (Past Masters (Vols. 1 & 2 / Remastered), 1988)
  Back In The U.S.S.R. → 'Back In The U.S.S.R. - Remastered' by 'The Beatles' (The Beatles (Remastered), 1968)
  Blackbird → 'Blackbird - Remastered' by 'The Beatles' (The Beatles (Remastered), 1968)
  Bohemian Rhapsody → 'Bohemian Rhapsody - Remastered 2011' by 'Queen' (A Night At The Opera (Deluxe Edition 2011 Remaster), 1975)
  Can_t Buy Me Love → 'Can't Buy Me Love - Remastered' by 'The Beatles' (The Beatles 1962 - 1966 (Remastered), 1964)
  Crazy Little Thing Called Love → 'Crazy Little Thing Called Love - Remastered 2011' by 'Queen' (The Game (2011 Remaster), 1980)
  Day Tripper → 'Day Tripper - Remast